# ЛР2. Проведение экспериментов по настройке модели
Задача — регрессия: предсказание `Selling_Price` автомобиля. Сравним базовую модель, генерацию признаков, экспертный отбор и подбор гиперпараметров. Оценка качества — MAE, MAPE, MSE; основной критерий сравнения — минимальная MAE. Перед запуском должен работать MLflow на http://127.0.0.1:5000.

## 1. Загрузка и разделение данных (пункты 1–4)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, PolynomialFeatures, KBinsDiscretizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from importlib.metadata import version

root = Path.cwd() if (Path.cwd() / 'data').is_dir() else Path.cwd().parent
research = root / 'research'
research.mkdir(exist_ok=True)
df = pd.read_pickle(root / 'data' / 'clean_dataset.pkl')
num_cols = ['Year', 'Present_Price', 'Driven_kms', 'Owner']
cat_cols = ['Car_Name', 'Fuel_Type', 'Selling_type', 'Transmission']
X = df[num_cols + cat_cols].copy()
y = df['Selling_Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print('Всего:', len(df), 'Train:', len(X_train), 'Test:', len(X_test))
X_train.head()

Всего: 299 Train: 224 Test: 75


,Year,Present_Price,Driven_kms,Owner,Car_Name,Fuel_Type,Selling_type,Transmission
225,2011,4.430,57000,0,i10,Petrol,Dealer,Manual
78,2012,30.610,89000,0,fortuner,Diesel,Dealer,Automatic
66,2010,20.450,59000,0,fortuner,Diesel,Dealer,Manual
296,2009,11.000,87934,0,city,Petrol,Dealer,Manual
192,2008,0.787,50000,0,Hero CBZ Xtreme,Petrol,Individual,Manual


## 2. Baseline (пункты 5–6)
StandardScaler масштабирует числа, OrdinalEncoder заменяет категории числами. Неизвестные категории кодируются как -1. RandomForestRegressor усредняет ответы деревьев. Pipeline объединяет подготовку и модель; преобразования обучаются только на train.

In [2]:
def base_transform():
    return ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)
    ])

def forest():
    return RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2)

def metrics(model):
    pred = model.predict(X_test)
    return {'mae': mean_absolute_error(y_test, pred),
            'mape': mean_absolute_percentage_error(y_test, pred),
            'mse': mean_squared_error(y_test, pred)}

baseline = Pipeline([('transform', base_transform()), ('regression', forest())])
baseline.fit(X_train, y_train)
metrics(baseline)

{'mae': 1.0879306666666673,
 'mape': 0.3060351112628624,
 'mse': 7.446190525066671}

## 3. MLflow (пункты 7–9)
Experiment объединяет прогоны (run). Логируем параметры, метрики, модель, сигнатуру, пример входа и файлы. В реестре сохраняем версии моделей.

In [3]:
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('Car price regression')
model_name = 'CarPriceRandomForest'
client = MlflowClient()
results = {}
models = {}
feature_lists = {}
feature_files = {}
model_requirements = [f'{p}=={version(p)}' for p in ['numpy', 'pandas', 'scikit-learn', 'scipy', 'cloudpickle']]

def save_lines(filename, values):
    path = research / filename
    path.write_text('\n'.join(map(str, values)) + '\n', encoding='utf-8')
    return path

def log_model(name, model, features, files=(), register=False, evaluate=True):
    with mlflow.start_run(run_name=name) as run:
        params = model.named_steps['regression'].get_params()
        mlflow.log_params(params)
        mlflow.log_param('feature_count', len(features))
        if evaluate:
            score = metrics(model)
            mlflow.log_metrics(score)
            results[name] = score
            models[name] = model
            feature_lists[name] = list(features)
            feature_files[name] = list(files)
        example = X_train.head(5)
        signature = infer_signature(example, model.predict(example))
        info = mlflow.sklearn.log_model(model, artifact_path='model',
            signature=signature, input_example=example,
            pip_requirements=model_requirements)
        mlflow.log_artifact(str(root / 'requirements.txt'))
        for path in files:
            mlflow.log_artifact(str(path))
        run_id = run.info.run_id
    registered = mlflow.register_model(info.model_uri, model_name) if register else None
    print(name, 'run_id:', run_id)
    if evaluate:
        print(results[name])
    return run_id, registered

base_names = baseline.named_steps['transform'].get_feature_names_out()
base_run, base_version = log_model('baseline', baseline, base_names, register=True)

2026/11/23 22:24:43 INFO mlflow.tracking.fluent: Experiment with name 'Car price regression' does not exist. Creating a new experiment.
/home/ubuntu/my_proj1/.venv_my_proj1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/23 22:24:52 INFO mlflow.tracking._tracking_service.client: 🏃 View run baseline

baseline run_id: 0f8f74ad30514f7c8a96c7080594aae2
{'mae': 1.0879306666666673, 'mape': 0.3060351112628625, 'mse': 7.446190525066671}


Created version '1' of model 'CarPriceRandomForest'.


## 4. Генерация признаков sklearn (пункт 10)
PolynomialFeatures создаёт квадраты и произведения цены и пробега; после этого новые числа масштабируются. KBinsDiscretizer делит год выпуска и пробег на интервалы. Сохраняем также исходные признаки.

In [4]:
fe_transform = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('poly', Pipeline([('poly', PolynomialFeatures(degree=2, include_bias=False)),
                       ('scale', StandardScaler())]), ['Present_Price', 'Driven_kms']),
    ('bins', KBinsDiscretizer(n_bins=3, encode='onehot-dense', strategy='uniform'), ['Year', 'Driven_kms'])
])
X_train_fe_sklearn = X_train.copy()
X_train_fe_sklearn = fe_transform.fit_transform(X_train_fe_sklearn)
fe_names = fe_transform.get_feature_names_out()
features_path = save_lines('features_sklearn.txt', fe_names)
print(fe_names)
print('Количество признаков:', len(fe_names))
fe_model = Pipeline([('transform', clone(fe_transform)), ('regression', forest())])
fe_model.fit(X_train, y_train)
fe_run, _ = log_model('feature_engineering', fe_model, fe_names, [features_path])

['num__Year' 'num__Present_Price' 'num__Driven_kms' 'num__Owner'
 'cat__Car_Name' 'cat__Fuel_Type' 'cat__Selling_type' 'cat__Transmission'
 'poly__Present_Price' 'poly__Driven_kms' 'poly__Present_Price^2'
 'poly__Present_Price Driven_kms' 'poly__Driven_kms^2' 'bins__Year_0.0'
 'bins__Year_1.0' 'bins__Year_2.0' 'bins__Driven_kms_0.0'
 'bins__Driven_kms_1.0' 'bins__Driven_kms_2.0']
Количество признаков: 19


/home/ubuntu/my_proj1/.venv_my_proj1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/23 22:29:15 INFO mlflow.tracking._tracking_service.client: 🏃 View run feature_engineering at: http://127.0.0.1:5000/#/experiments/1/runs/8f4c59118ded4e688b3aa7333fc81032.
2026/11/23 22:29:15 INFO mlflow.tracking._t

feature_engineering run_id: 8f4c59118ded4e688b3aa7333fc81032
{'mae': 1.158245333333334, 'mape': 0.3202773661193149, 'mse': 8.578272258666669}


## 5. Экспертный отбор признаков (пункт 12)
Методические указания разрешают экспертный отбор. По EDA оставляем стоимость, год выпуска, пробег, вид топлива, коробку передач, а также несколько созданных признаков. Отбор не использует тестовые ответы. Индексы соответствуют выходу ColumnTransformer.

In [5]:
wanted = ['num__Year', 'num__Present_Price', 'num__Driven_kms',
          'cat__Fuel_Type', 'cat__Transmission',
          'poly__Present_Price^2', 'poly__Present_Price Driven_kms',
          'bins__Year_2.0']
selected_indices = [i for i, name in enumerate(fe_names) if name in wanted]
selected_names = fe_names[selected_indices]
assert 0.2 <= len(selected_names) / len(fe_names) <= 0.7
print('Выбрано:', len(selected_names), 'из', len(fe_names))
print(selected_names, selected_indices)
names_path = save_lines('selected_features.txt', selected_names)
indices_path = save_lines('selected_indices.txt', selected_indices)
selector = ColumnTransformer([('selected', 'passthrough', selected_indices)], remainder='drop')
selected_model = Pipeline([('transform', clone(fe_transform)), ('selection', selector), ('regression', forest())])
selected_model.fit(X_train, y_train)
selected_run, _ = log_model('feature_selection', selected_model, selected_names,
                            [features_path, names_path, indices_path])

Выбрано: 8 из 19
['num__Year' 'num__Present_Price' 'num__Driven_kms' 'cat__Fuel_Type'
 'cat__Transmission' 'poly__Present_Price^2'
 'poly__Present_Price Driven_kms' 'bins__Year_2.0'] [0, 1, 2, 5, 7, 10, 11, 15]


/home/ubuntu/my_proj1/.venv_my_proj1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/23 22:31:36 INFO mlflow.tracking._tracking_service.client: 🏃 View run feature_selection at: http://127.0.0.1:5000/#/experiments/1/runs/de2cfa5c51ed4143a24d931a978db1f8.
2026/11/23 22:31:36 INFO mlflow.tracking._tra

feature_selection run_id: de2cfa5c51ed4143a24d931a978db1f8
{'mae': 1.2235200000000013, 'mape': 0.3234963190913637, 'mse': 9.212178608266672}


## 6. Подбор гиперпараметров GridSearchCV (пункт 14)
Выбираем лучшую конфигурацию среди предыдущих экспериментов по MAE. Проверяем 12 комбинаций настроек: число деревьев, глубина, доля признаков. CV делит обучающую выборку на три части. `neg_mean_absolute_error` максимизируется sklearn, что равносильно минимизации MAE. Тестовая выборка не используется внутри поиска.

In [6]:
best_before = min(results, key=lambda name: results[name]['mae'])
print('Лучший вариант до настройки:', best_before)
param_grid = {
    'regression__n_estimators': [50, 100],
    'regression__max_depth': [5, 10, None],
    'regression__max_features': [0.5, 1.0]
}
search = GridSearchCV(clone(models[best_before]), param_grid,
                      scoring='neg_mean_absolute_error', cv=3, n_jobs=1)
search.fit(X_train, y_train)
print('Лучшие параметры:', search.best_params_)
print('MAE на CV:', -search.best_score_)
print('Комбинаций:', len(search.cv_results_['params']))
grid_results = research / 'grid_results.txt'
grid_results.write_text(pd.DataFrame(search.cv_results_)[['params', 'mean_test_score', 'rank_test_score']].to_string(index=False), encoding='utf-8')
tuned_run, tuned_version = log_model('grid_search', search.best_estimator_,
    feature_lists[best_before], feature_files[best_before] + [grid_results], register=True)

Лучший вариант до настройки: baseline
Лучшие параметры: {'regression__max_depth': 10, 'regression__max_features': 1.0, 'regression__n_estimators': 100}
MAE на CV: 0.7091743660310911
Комбинаций: 12


/home/ubuntu/my_proj1/.venv_my_proj1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/23 22:33:53 INFO mlflow.tracking._tracking_service.client: 🏃 View run grid_search at: http://127.0.0.1:5000/#/experiments/1/runs/bf0cb578770a4e0e9fb8f91e26ebdb1f.
2026/11/23 22:33:53 INFO mlflow.tracking._tracking_

grid_search run_id: bf0cb578770a4e0e9fb8f91e26ebdb1f
{'mae': 1.1693277905575912, 'mape': 0.3255918849315695, 'mse': 9.10846651454993}


Created version '2' of model 'CarPriceRandomForest'.


## 7. Сравнение и Production (пункт 16)
Сравниваем MAE, MAPE и MSE на отложенной выборке. Выбираем минимальную MAE, затем переобучаем выбранный Pipeline на всех очищенных данных. Метрики финальной модели на обучающих данных не считаем. Сохраняем её в реестре и переводим в Production.

In [7]:
comparison = pd.DataFrame(results).T.sort_values('mae')
display(comparison)
best_name = comparison.index[0]
production = clone(models[best_name]).fit(X, y)
production_features = feature_lists[best_name]
production_path = save_lines('production_features.txt', production_features)
production_run, production_version = log_model('production', production,
    production_features, feature_files[best_name] + [production_path], register=True, evaluate=False)
client.transition_model_version_stage(name=model_name, version=production_version.version,
                                     stage='Production', archive_existing_versions=True)
client.set_model_version_tag(model_name, production_version.version, 'Production', 'true')
print('Лучший эксперимент:', best_name)
print('Production run_id:', production_run)
print('Версия:', production_version.version)
mlflow.artifacts.download_artifacts(artifact_uri=f'runs:/{production_run}/model/MLmodel', dst_path=str(research))

,mae,mape,mse
baseline,1.087931,0.306035,7.446191
feature_engineering,1.158245,0.320277,8.578272
grid_search,1.169328,0.325592,9.108467
feature_selection,1.223520,0.323496,9.212179


/home/ubuntu/my_proj1/.venv_my_proj1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/23 22:35:18 INFO mlflow.tracking._tracking_service.client: 🏃 View run production at: http://127.0.0.1:5000/#/experiments/1/runs/363668ff43144c9c83afb9c3cb979316.
2026/11/23 22:35:18 INFO mlflow.tracking._tracking_s

production run_id: 363668ff43144c9c83afb9c3cb979316
Лучший эксперимент: baseline
Production run_id: 363668ff43144c9c83afb9c3cb979316
Версия: 3


'/home/ubuntu/my_proj1/research/MLmodel'

## 8. Результаты для README (пункты 17–19)
Сохраняем реальные результаты экспериментов. Скриншоты интерфейса MLflow нужно сделать отдельно: список прогонов и графики метрик, затем список версий модели.

In [ ]:
readme = root / 'README.md'
marker = '## ЛР2. Результаты исследования'
score = results[best_name]
params = production.named_steps['regression'].get_params()
section = f"""{marker}

Запуск MLflow из корня проекта после активации виртуального окружения: `sh mlflow/start_mlflow.sh`.
Интерфейс: http://127.0.0.1:5000. Блокнот: `research/research.ipynb`.

Сравнение моделей на тестовой выборке:

| Эксперимент | MAE | MAPE | MSE |
|---|---:|---:|---:|
"""
for name, row in comparison.iterrows():
    section += f"| {name} | {row.mae:.4f} | {row.mape:.4f} | {row.mse:.4f} |\n"
section += f"""
Лучший эксперимент по MAE: `{best_name}`. MAPE приведена в долях (для процентов умножить на 100).
Параметры: n_estimators={params['n_estimators']}, max_depth={params['max_depth']}, max_features={params['max_features']}.

Признаки после преобразований и отбора: {', '.join(map(str, production_features))}.
Входные столбцы исходного Pipeline: {', '.join(X.columns)}.

Финальная модель переобучена на всей выборке и зарегистрирована в Production.
Run ID: `{production_run}`. Версия: `{production_version.version}`.
"""
old = readme.read_text(encoding='utf-8') if readme.exists() else ''
readme.write_text(old.split(marker)[0].rstrip() + '\n\n' + section, encoding='utf-8')
print(section)